# **Generating adversarial noise examples using FGSM**
By Charlot Eberlein, Loughborough University • 11.01.26 • Updated 20.08.2026 •
[LinkedIn](https://www.linkedin.com/in/charloteberlein/) • 
[Portfolio](https://charloteberlein.github.io) • 
[GitHub](https://github.com/charloteberlein)

In [ ]:
%%capture
import tensorflow as tf
import keras as k
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from ipywidgets import widgets as w

### **Step 1:** Load the MNIST dataset

In [ ]:
(train_im, train_l), (test_im, test_l) = k.datasets.mnist.load_data()

train_im = train_im.reshape((60000,28,28,1)).astype("float32")/255
test_im = test_im.reshape((10000,28,28,1)).astype("float32")/255

Let's quickly make sure it works using matplotlib:

In [ ]:
SIZE = 6; assert SIZE > 1
fig, ax = plt.subplots(SIZE, SIZE)
fig.suptitle(f"First {SIZE**2} MNIST digits")
for i in range(SIZE):
    for j in range(SIZE):
        p = SIZE*i + j
        ax[i][j].imshow(np.reshape(train_im[p], (28,28)), cmap="gray")
        ax[i][j].set_title(train_l[p], y=-0.1, x=0.9, color="white")
        ax[i][j].set_axis_off()
plt.tight_layout()
plt.show()

If the dataset has been loaded correctly, the result should look something like this:

![Grid of digits](resources/first_36_mnist.png)

Great. Now, let's move on to the next step.

### **Step 2:** Build a Convolutional Neural Network (CNN) with TensorFlow
This architecture is similar to one I've used before in another project, with a bit of trial-and-error to produce the best results.
- **Input layer**
    - With added data augmentation (Random Elastic Transform)

- **Convolutional layer** (ReLU activation)
    - Batch normalisation

- **Max-pooling layer**

- **Convolutional layer** (ReLU activation)
    - Batch normalisation

- **Max-pooling layer**

- **Fully-connected layers**

- **Output layer** (Softmax activation)

**Random elastic transform** is a data augmentation method I think is best described like the "liquify" tool in photoshop. Here, we have an intensity of 10% which is randomly applied in 5% of the training data.

**Batch normalisation** normalises layer inputs to a mean of 0 and standard deviation of 1, which improves training speed and stability.

In [ ]:
# Augmented model
augmented_model = k.Sequential([
    k.Input(shape=(28,28,1)),
    k.layers.RandomElasticTransform(factor=0.05, scale=0.1),

    k.layers.Conv2D(32, activation="relu", padding="same", kernel_size=5),
    k.layers.BatchNormalization(),
    k.layers.MaxPool2D(pool_size=(2,2), strides=4),

    k.layers.Conv2D(64, activation="relu", padding="same", kernel_size=5),
    k.layers.BatchNormalization(),
    k.layers.MaxPool2D(pool_size=(2,2), strides=4),

    k.layers.Flatten(),
    k.layers.Dense(128, activation="relu"),
    k.layers.Dense(10, activation="softmax"),
])

augmented_model.compile(
    optimizer="sgd",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

We're also going to make an identical version, just without the data augmentation and batch normalisation, just to compare.

In [ ]:
# Normal model
normal_model = k.Sequential([
    k.Input(shape=(28,28,1)),

    k.layers.Conv2D(32, activation="relu", padding="same", kernel_size=5),
    k.layers.MaxPool2D(pool_size=(2,2), strides=4),

    k.layers.Conv2D(64, activation="relu", padding="same", kernel_size=5),
    k.layers.MaxPool2D(pool_size=(2,2), strides=4),

    k.layers.Flatten(),
    k.layers.Dense(128, activation="relu"),
    k.layers.Dense(10, activation="softmax"),
])

normal_model.compile(
    optimizer="sgd",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

Finally, we're going to make a really simple model with just one convolutional layer. I'm going to increase the stride size here because I want this to be intentionally worse, for demonstration purposes.

In [ ]:
# Simple model (with higher stride)
simple_model = k.Sequential([
    k.Input(shape=(28,28,1)),

    k.layers.Conv2D(32, activation="relu", padding="same", kernel_size=5),
    k.layers.MaxPool2D(pool_size=(2,2), strides=8),
    
    k.layers.Flatten(),
    k.layers.Dense(10, activation="softmax"),
])

simple_model.compile(
    optimizer="sgd",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

#### **Optional:** Train the models
Because this is a really simple problem, training should only take a minute or two for each model we've just defined. However, I've prepared some weights earlier, so this step is optional.

You can really see how using data augmentation paid off in our first model, because we have a high degree of accuracy:\
• **Training accuracy:** 99.99%\
• **Validation accuracy:** 98.76%\
• **Testing accuracy:** 98.65%

In [ ]:
# Prepare the data
from sklearn.model_selection import train_test_split

# 80-20 train-validation split
train_im_split, val_im, train_l_split, val_l = train_test_split(
    train_im, train_l, test_size=0.2, random_state=42
)

train_l_cat = k.utils.to_categorical(train_l_split)
val_l_cat = k.utils.to_categorical(val_l)
test_l_cat = k.utils.to_categorical(test_l)

In [ ]:
# Augmented model
result = augmented_model.fit(
    train_im_split, train_l_cat,
    batch_size=64,
    epochs=16,
    verbose=1,
    validation_data=(val_im, val_l_cat)
)
augmented_model.save_weights("resources/mnist-augmented.weights.h5")

In [ ]:
# Normal model
result = normal_model.fit(
    train_im_split, train_l_cat,
    batch_size=64,
    epochs=16,
    verbose=1,
    validation_data=(val_im, val_l_cat)
)
normal_model.save_weights("resources/mnist-norm.weights.h5")

In [ ]:
# Simple model (with higher stride)
result = simple_model.fit(
    train_im_split, train_l_cat,
    batch_size=64,
    epochs=16,
    verbose=1,
    validation_data=(val_im, val_l_cat)
)
simple_model.save_weights("resources/mnist-weak.weights.h5")

In [ ]:
# Display training results (loss-epochs curves)

fig, (ax1, ax2) = plt.subplots(1, 2)

ax1.plot(result.history["loss"], color="b", label="Training loss")
ax1.plot(result.history["val_loss"], color="r", label="Validation loss")
ax1.set_title("Loss")
ax1.set_xlabel("Epochs")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(result.history["accuracy"], color="b", label="Training accuracy")
ax2.plot(result.history["val_accuracy"], color="r", label="Validation accuracy")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epochs")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
MODEL = normal_model

test_loss, test_accuracy, = MODEL.evaluate(test_im, test_l_cat)
print(f"Test accuracy: {test_accuracy:.4f}")

#### Training results

Simple model | Normal model | Augmented model
--|--|--
![Simple loss-epochs curves](resources/simple_curves.png) | ![Normal loss-epochs curves](resources/normal_curves.png) | ![Augmented loss-epochs curves](resources/augmented_curves.png)

Note that the distance between the curves on the augmented model curve is because data augmentation was added to the training data, but not to the testing data.

#### Sanity check
Finally, let's do one last sanity check to make sure our models are working as they should be. We'll do this by checking a subset of the testing images ourselves.

In [ ]:
# Load models and get predictions
augmented_model.load_weights("resources/mnist-augmented.weights.h5")
normal_model.load_weights("resources/mnist-norm.weights.h5")
simple_model.load_weights("resources/mnist-weak.weights.h5")

In [ ]:
# Plot one to test
TEST_MODEL = augmented_model

pred = TEST_MODEL.predict(test_im, verbose=0)

fig, ax = plt.subplots(
    nrows=3, ncols=5, layout="tight", figsize=(8,3),
    gridspec_kw={"height_ratios":[1,0.1,1]}
)

for i in range(5):
    ax[0,i].imshow(test_im[i], cmap="gray")
    ax[0,i].set_axis_off()

for i in range(5):
    ax[1,i].text(
        0.5, 0.5, f"Prediction: {np.argmax(pred[i])}",
        horizontalalignment="center", verticalalignment="center"
    )
    ax[1,i].set_axis_off()

for i in range(5):
    ax[2,i].step(np.arange(0,10), pred[i], c="blue")

plt.show()

If everything is working correctly, you should see the following plot:

![Plot of digits and predictions](resources/mnist_eval.png)

Great! Now, we're ready to get to the fun part: FGSM!

### **Step 3:** Use FGSM to generate Adversarial Noise Examples
We're going to use some tools from TensorFlow to help us implement FGSM. Essentially, what we're doing here is looking at the gradients of the loss function with respect to our target image and input class. This creates some noise which we're going to subtract from the input image, which will (depending on the model) cause the input image to be misclassified as the target class. We are pushing the gradients **towards** the target class.

I've implemented this with an interactive GUI which you can use to explore the different models and compare epsilon (noise opacity) values.
- **Simple model:** a single convolutional layer, which is the most susceptible to manipulation. This also has a larger stride to make it even worse. Almost all images can be successfully misclassified using FGSM.
- **Normal model:** two convolutional layers, but without any data augmentation. Not as vulnerable as the simple model, but still easy to deceive.
- **Augmented model:** same as the normal model, but with added data augmentation (resulting in 99% validation accuracy). More difficult to misclassify to a specific target class.

Compare the gradient noise images. What do you notice?

In [ ]:
def _generate_adversarial_noise(
    model: k.Sequential, im: tf.Tensor, target: tf.Tensor, epsilon: float
) -> tf.Tensor:
    with tf.GradientTape() as tape:
        tape.watch(im)
        pred = model(im) # forward pass
        target = tf.reshape(target, pred.shape)
        loss = -k.losses.categorical_crossentropy(target, pred)
    return epsilon * tf.sign(tape.gradient(loss,im)) # backward pass

def main(
    model: k.Sequential = [augmented_model, normal_model, simple_model],
    epsilon: float = 0.3, input_index: int = 9, target_index: int = 13
) -> None:
    # np -> tf
    im_tensor = tf.convert_to_tensor(test_im[input_index], dtype=tf.float32)
    im_tensor = tf.reshape(im_tensor, [1,28,28,1])
    target_tensor = tf.one_hot(test_l[target_index], depth=10)
    perturbations = _generate_adversarial_noise(
        model, im_tensor, target_tensor, epsilon
    )[0]

    pred_norm = model.predict(test_im, verbose=0)
    adversarial_example = test_im + perturbations
    adversarial_example = tf.clip_by_value(adversarial_example, 0, 1)
    pred_adversarial = model.predict(adversarial_example, verbose=0)

    # plot graph
    fig, ax = plt.subplots(
        nrows=2, ncols=4, figsize=(8,2), gridspec_kw={"height_ratios":[1, 0.1]}
    )

    ax[0,0].imshow(test_im[input_index], cmap="gray_r")
    ax[0,1].imshow(test_im[target_index], cmap="gray_r")
    ax[0,2].imshow(perturbations, cmap="gray_r")
    ax[0,3].imshow(adversarial_example[input_index], cmap="gray_r")

    ax[1,0].text(
        0.5, 0.5, f"Prediction: {np.argmax(pred_norm[input_index])} "
        f"({format(np.max(pred_norm[input_index])*100, ".1f")}%)",
        horizontalalignment="center", verticalalignment="center"
    )
    ax[1,1].text(
        0.5, 0.5, f"Target class: {test_l[target_index]}",
        horizontalalignment="center", verticalalignment="center"
    )
    ax[1,2].text(
        0.5, 0.5, f"FGSM Pattern ({epsilon*100}%)",
        horizontalalignment="center", verticalalignment="center"
    )
    ax[1,3].text(
        0.5, 0.5, f"Prediction: {np.argmax(pred_adversarial[input_index])}"
        f" ({format(np.max(pred_adversarial[input_index])*100, ".1f")}%)",
        horizontalalignment="center", verticalalignment="center"
    )

    for ax0 in ax:
        for ax1 in ax0:
            ax1.set_axis_off()

    plt.show()

interact(main,
    model = w.Dropdown(
        options=[
            ("Augmented", augmented_model), ("Normal", normal_model),
            ("Simple", simple_model)
        ],
        description="Model:"
    ),
    epsilon = w.FloatSlider(
        value=0.1, min=0.0, max=1.0, step=0.05, description="Epsilon:",
        continuous_update=False
    ),
    input_index = w.BoundedIntText(
        value=0, min=0, max=10000, description="Input Index:"
    ),
    target_index = w.BoundedIntText(
        value=4, min=0, max=10000, description="Target Index:"
    )
)